# TUTORIAL: Real-time digital twin of a hydrogen-based annular combustor
 

We can now put everything we have learned together. Data assimilation using real experimental data

We develop a digital twin of the hydrogen-based annular combustor. We want to estimate the LOM parameters and states from raw experimental data from microphones.  Because the raw data may be biased, we need to model the bias in both, measurement data and model.


In [ ]:
from romda.utils import set_working_directories, get_annular_data

data_folder, results_folder, figs_folder = set_working_directories('annular')
get_annular_data(data_folder) # Download the data if not already present

## 1. Load data 
Create the reference truth and the observations.



In [ ]:
from romda.models.physical import Annular
from romda.observations import Observations
import os

ER = 0.4875 + 0.025 # 0.4875 + np.arange(0, 4) * 0.025

t_start = Annular.t_transient
t_stop = t_start + Annular.t_CR * 15


truth = Observations(model = os.path.join(data_folder, 'ER_{}'.format(ER)),
                     t_start = t_start,
                     t_stop = t_stop,
                     Nt_obs = 35,
                     t_max = t_stop + Annular.t_transient,
                     add_noise = False
                     )


In [ ]:
Observations.plot_truth(truth, Nq=4, fig_width=12, window=0.025, f_max=10000)

## 2. Define the forecast model
This is the physical model which we will use to model the true data.
Here, we select the filter parameters and create ensemble

*The function ```create_ensemble``` consists of* 

```
alpha0_mean = dict()
for alpha, lims in alpha0.items():
    alpha0_mean[alpha] = 0.5 * (lims[0] + lims[1])

ensemble = Annular(**alpha0_mean)

filter_params = dict(m= 20, 
                     std_psi=0.3,
                     std_a=alpha0)

# Forecast model to initialise the ensemble after transient
state, t_ = ensemble.time_integrate(int(ensemble.t_CR / ensemble.dt))
ensemble.update_history(state[-1], reset=True)

ensemble.init_ensemble(**filter_params)
ensemble.close()
```

In [ ]:
from romda.ensemble import Ensemble
from romda.data_assimilation import rBA_EnKF  # try also 'EnKF', 'EnSRKF'
import numpy as np

alpha0 = dict(nu=(-15., 30.),
              c2beta=(10, 50),
              kappa=(1.E-4, 2.E-4),
              epsilon=(5e-3, 8e-3),
              omega=(1090 * 2 * np.pi, 1095 * 2 * np.pi),
              theta_b=(0.5, 0.7),
              theta_e=(0.4, 0.6)
              )

ensemble = Ensemble(parent_model=Annular(dt=truth.dt),
                    da_method=rBA_EnKF,
                    m=20,
                    std_phi=0.3,
                    std_alpha=alpha0,
                    distribution_alpha='uniform',
                    )

In [ ]:
# Visualize ensemble initialization
ensemble.visualize_state(reference_a={'kappa': 1e-4, 'omega': 2 * np.pi, 'epsilon': 1e-3})

## 4. Train an ESN to model the model bias
The procedure is the following

&emsp; i. Initialise ESN Bias class object
&emsp; ii. Create synthetic bias to use as training data 
&emsp; iii. Train the ESN
&emsp; iv. Create washout data

<br>

**4.1. Initialise the ESN**

In [ ]:
from romda.bias_estimators import ESN_bias
import numpy as np

training_data_filename = f'{results_folder}/ESN_train_data_annular_raw'

bias_estimator = ESN_bias(rom=ensemble.model,
                          reference_data=truth,
                          training_data_filename=training_data_filename,
                          upsample=5,
                          N_units=50,
                          N_wash=10,
                          t_train=ensemble.model.t_transient / 3.,
                          t_test=ensemble.model.t_CR * 2,
                          t_val=ensemble.model.t_CR * 2,
                          # Training data generation options
                          augment_data=True,
                          biased_observations=True,
                          correlation_based_training=True,
                          N_folds=4,
                          L=20,
                          std_alpha=alpha0,
                          # Hyperparameter search ranges
                          rho_range=(0.5, 1.),
                          sigma_in_range=(np.log10(1e-5), np.log10(1e1)),
                          tikh_range=[1e-12, 1e-9],
                          N_ens=ensemble.m,
                          )


**4.2 Create training data**

The details of the code inside ```create_bias_training_dataset()``` function is explained in the tutorial ```Class_Bias.ipynb```.

**4.3. Train the ESN**

The training convergence, hyperparameter optimization and testing results are saved in a pdf file in *figs_ESN* folder.

**4.4. Create washout data**

We retrieve from the raw data a ```N_wash``` number of observations to use for initialising the ESN, i.e., to perform the washout. 
The ESN initialization must be before the fist observation.

```
from create import create_washout
wash_t, wash_obs = create_washout(ensemble.bias, t=t_true, y_raw=y_raw)
```

In [ ]:
ensemble_BA = ensemble.copy()
ensemble_BA.bias = bias_estimator.copy()

## 5. Apply data assimilation
We now have all the ingredients to start our data assimilation algorithm.

In [ ]:
# Observation error covariance matrix
std_obs = 0.05
Cdd = np.diag(std_obs * np.ones(ensemble.model.Nq)) * np.max(abs(truth.y_obs), axis=0) ** 2

t_extra = ensemble.model.t_CR * 10

out = []
ks = [0, 5.]  # bias regularization factors

for kk in ks:
    ens = ensemble_BA.copy()
    ens.regularization_factor = kk
    ens.filter.gamma = kk

    for d, t_d in zip(truth.y_obs, truth.t_obs):
        ens.forecast_step(t_end=t_d)
        ens.analysis_step(d=d, Cdd=Cdd.copy())

    ens.forecast_step(t_end=truth.t_obs[-1] + t_extra, close=True)
    out.append(ens)

In [ ]:
for ens in out:
    ens.visualize_history(truth=truth, plot_members=False, dims=[0, 1])
    ens.bias.visualize_bias_and_innovations(plot_members=True)